# 🎙️ Speech-to-Reasoning Pipeline
## Whisper ASR → Quantized LLM (Unsloth Dynamic 4-bit)

This notebook implements an end-to-end pipeline:
1. **Audio Input** → Whisper transcribes speech to text
2. **Transcribed Text** → Quantized Reasoning LLM generates a logical answer

### Architecture Overview
```
Audio File (WAV/MP3)
        │
        ▼
  ┌─────────────┐
  │  OpenAI     │
  │  Whisper    │  ← ASR Model (float16 / int8)
  │  (base/small)│
  └──────┬──────┘
         │  Transcribed Text
         ▼
  ┌─────────────┐
  │  Prompt     │
  │  Template   │  ← Reasoning instruction wrapper
  └──────┬──────┘
         │
         ▼
  ┌─────────────┐
  │  Unsloth    │
  │  Dynamic    │  ← 4-bit quantized LLM (Qwen / Llama)
  │  4-bit LLM  │
  └──────┬──────┘
         │  Reasoned Answer
         ▼
    Final Output
```

> **Runtime**: Select **GPU (T4 or better)** before running. `Runtime → Change runtime type → T4 GPU`

---
## 📦 Step 1: Install Dependencies

In [ ]:
# ── Install all required packages ──────────────────────────────────────────
# Unsloth (handles quantized model loading + LoRA efficiently)
!pip install unsloth --quiet

# OpenAI Whisper for ASR
!pip install openai-whisper --quiet

# Audio utilities
!pip install soundfile librosa pydub --quiet

# ffmpeg for audio decoding (required by Whisper)
!apt-get install -y ffmpeg --quiet

# BitsAndBytes for quantization fallback support
!pip install bitsandbytes --quiet

# Transformers & accelerate
!pip install transformers accelerate --quiet

print("✅ All dependencies installed.")

---
## 🔧 Step 2: Imports & GPU Verification

In [ ]:
import os
import gc
import time
import torch
import whisper
import numpy as np
import soundfile as sf
import librosa
from pathlib import Path
from IPython.display import Audio, display, Markdown

# ── GPU Check ──────────────────────────────────────────────────────────────
print("=" * 55)
print("  🖥️  SYSTEM DIAGNOSTICS")
print("=" * 55)

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    total_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"  GPU  : {gpu_name}")
    print(f"  VRAM : {total_mem:.1f} GB")
    DEVICE = "cuda"
    COMPUTE_TYPE = "float16"   # Whisper precision on GPU
else:
    print("  ⚠️  No GPU found — falling back to CPU (slow)")
    DEVICE = "cpu"
    COMPUTE_TYPE = "int8"      # Use int8 on CPU to save RAM

print(f"  Device        : {DEVICE}")
print(f"  PyTorch       : {torch.__version__}")
print("=" * 55)

---
## 🎵 Step 3: Prepare Sample Audio
We programmatically generate a sample WAV file using gTTS (text-to-speech) so the pipeline is fully self-contained and runnable without any external upload.

In [ ]:
# ── Install gTTS to synthesize a demo audio query ─────────────────────────
!pip install gtts --quiet

from gtts import gTTS
import IPython.display as ipd

# ── Define sample spoken query ─────────────────────────────────────────────
SAMPLE_QUERY = (
    "If a train travels 120 kilometers in 2 hours, "
    "and then 180 kilometers in the next 3 hours, "
    "what is the average speed of the train for the entire journey? "
    "Please reason through this step by step."
)

AUDIO_PATH = "sample_query.mp3"

print(f"📝 Synthesizing audio for query:")
print(f'   "{SAMPLE_QUERY}"')

tts = gTTS(text=SAMPLE_QUERY, lang="en", slow=False)
tts.save(AUDIO_PATH)

print(f"✅ Audio saved → {AUDIO_PATH}")

# Play the audio inline
print("\n🔊 Preview the audio:")
display(Audio(AUDIO_PATH))

---
## 🗣️ Step 4: Load Whisper ASR Model & Transcribe

In [ ]:
# ── Configuration ──────────────────────────────────────────────────────────
# Whisper model sizes: tiny | base | small | medium | large-v3
# For T4 (16GB VRAM), 'small' offers good accuracy with fast speed.
# Switch to 'medium' or 'large-v3' for higher accuracy on complex audio.
WHISPER_MODEL_SIZE = "small"   # Change to "medium" for better accuracy

print(f"⏳ Loading Whisper '{WHISPER_MODEL_SIZE}' on {DEVICE}...")
t0 = time.time()

whisper_model = whisper.load_model(
    WHISPER_MODEL_SIZE,
    device=DEVICE
)

print(f"✅ Whisper loaded in {time.time()-t0:.1f}s")

# ── Transcription ──────────────────────────────────────────────────────────
print(f"\n🎙️  Transcribing: {AUDIO_PATH} ...")
t1 = time.time()

transcription_result = whisper_model.transcribe(
    AUDIO_PATH,
    language="en",          # force English; remove for auto-detect
    fp16=(DEVICE == "cuda"), # use fp16 on GPU for speed
    verbose=False
)

TRANSCRIBED_TEXT = transcription_result["text"].strip()
detected_lang    = transcription_result.get("language", "en")

print(f"✅ Transcription complete in {time.time()-t1:.1f}s")
print(f"\n{'─'*55}")
print(f"  🌐 Detected Language : {detected_lang}")
print(f"  📝 Transcribed Text  :")
print(f"\n  {TRANSCRIBED_TEXT}")
print(f"{'─'*55}")

---
## 📊 Step 5: Inspect Whisper Segment-Level Output
Whisper returns per-segment timestamps — useful for long audio.

In [ ]:
segments = transcription_result.get("segments", [])

print(f"📊 Segment-level breakdown ({len(segments)} segments):")
print(f"{'─'*65}")
print(f"  {'#':<4} {'Start':>7} {'End':>7}   Text")
print(f"{'─'*65}")

for seg in segments:
    idx   = seg['id']
    start = seg['start']
    end   = seg['end']
    text  = seg['text'].strip()
    print(f"  {idx:<4} {start:>6.2f}s {end:>6.2f}s   {text}")

print(f"{'─'*65}")

# ── Free Whisper from GPU VRAM before loading the LLM ─────────────────────
print("\n🧹 Releasing Whisper from GPU memory...")
del whisper_model
gc.collect()
torch.cuda.empty_cache()

if DEVICE == "cuda":
    free_vram = (torch.cuda.mem_get_info()[0]) / 1e9
    print(f"✅ VRAM freed. Available: {free_vram:.2f} GB")

---
## 🤖 Step 6: Load Quantized Reasoning LLM via Unsloth

We load **Qwen2.5-7B-Instruct** (dynamic 4-bit) via Unsloth. 
Unsloth reduces VRAM by ~60% vs standard float16 loading and provides 2× faster inference.

> **Alternative models** (uncomment to switch):
> - `unsloth/Llama-3.2-3B-Instruct-bnb-4bit`
> - `unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit`
> - `unsloth/Qwen2.5-7B-Instruct-bnb-4bit`

In [ ]:
from unsloth import FastLanguageModel

# ── Model Configuration ────────────────────────────────────────────────────
MODEL_NAME   = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit"  # Dynamic 4-bit
MAX_SEQ_LEN  = 2048     # Max tokens (input + output combined)
DTYPE        = None     # Auto-detect: float16 on Ampere+, bfloat16 otherwise
LOAD_IN_4BIT = True     # Enable dynamic 4-bit quantization

print(f"⏳ Loading quantized model: {MODEL_NAME}")
print(f"   max_seq_length : {MAX_SEQ_LEN}")
print(f"   4-bit quant    : {LOAD_IN_4BIT}")
print(f"   dtype          : Auto")

t2 = time.time()

llm_model, tokenizer = FastLanguageModel.from_pretrained(
    model_name      = MODEL_NAME,
    max_seq_length  = MAX_SEQ_LEN,
    dtype           = DTYPE,
    load_in_4bit    = LOAD_IN_4BIT,
)

# ── Enable fast inference mode (disables gradient tracking) ───────────────
FastLanguageModel.for_inference(llm_model)

load_time = time.time() - t2
print(f"\n✅ Model loaded in {load_time:.1f}s")

if DEVICE == "cuda":
    used_vram = torch.cuda.memory_allocated() / 1e9
    print(f"   VRAM used by model : {used_vram:.2f} GB")

---
## 🧠 Step 7: Build the Reasoning Prompt
We wrap the transcribed text in a structured chat template designed to elicit step-by-step logical reasoning.

In [ ]:
# ── Prompt Engineering ──────────────────────────────────────────────────────
# The system message instructs the model to reason carefully.
# The user message contains the transcribed audio query.

SYSTEM_PROMPT = """You are an expert reasoning assistant. 
When given a question or problem, you:
1. Carefully read and understand the question.
2. Break it down into clear logical steps.
3. Solve each step explicitly, showing your work.
4. State the final answer clearly at the end.
Always be precise, structured, and thorough."""

# Build chat messages using Qwen's expected format
messages = [
    {"role": "system",    "content": SYSTEM_PROMPT},
    {"role": "user",      "content": TRANSCRIBED_TEXT},
]

# ── Apply tokenizer chat template ──────────────────────────────────────────
# This converts messages to the model-specific prompt format
formatted_prompt = tokenizer.apply_chat_template(
    messages,
    tokenize    = False,      # Return string first for inspection
    add_generation_prompt = True  # Append the assistant turn prefix
)

print("📋 Formatted prompt sent to LLM:")
print("─" * 55)
print(formatted_prompt)
print("─" * 55)

# ── Tokenize & count tokens ────────────────────────────────────────────────
input_ids = tokenizer(
    formatted_prompt,
    return_tensors = "pt"
).input_ids.to(DEVICE)

num_input_tokens = input_ids.shape[-1]
print(f"\n🔢 Input token count : {num_input_tokens}")

---
## ⚡ Step 8: Run Inference — Generate Reasoning Output

In [ ]:
# ── Generation Parameters ──────────────────────────────────────────────────
GEN_CONFIG = dict(
    max_new_tokens  = 512,    # Max tokens the model may generate
    temperature     = 0.3,    # Low temp → more deterministic reasoning
    top_p           = 0.9,    # Nucleus sampling
    top_k           = 50,     # Top-k filtering
    repetition_penalty = 1.1, # Discourage repetition
    do_sample       = True,   # Sampling (set False for greedy)
    use_cache       = True,   # KV cache for faster decode
    pad_token_id    = tokenizer.eos_token_id,
)

print("🤔 Generating reasoning response...")
print(f"   max_new_tokens : {GEN_CONFIG['max_new_tokens']}")
print(f"   temperature    : {GEN_CONFIG['temperature']}")

t3 = time.time()

with torch.no_grad():       # Disable gradient computation during inference
    output_ids = llm_model.generate(
        input_ids,
        **GEN_CONFIG
    )

inference_time = time.time() - t3

# ── Decode only the newly generated tokens ─────────────────────────────────
# Slice off input_ids to get only the assistant's response
generated_ids = output_ids[:, num_input_tokens:]
response = tokenizer.decode(
    generated_ids[0],
    skip_special_tokens = True
).strip()

num_output_tokens = generated_ids.shape[-1]
tokens_per_sec    = num_output_tokens / inference_time

print(f"\n✅ Generation complete!")
print(f"   Output tokens     : {num_output_tokens}")
print(f"   Inference time    : {inference_time:.2f}s")
print(f"   Throughput        : {tokens_per_sec:.1f} tokens/sec")

---
## 📤 Step 9: Display Full Pipeline Results

In [ ]:
# ── Final Pipeline Summary ─────────────────────────────────────────────────
SEPARATOR = "═" * 60
SEP_THIN  = "─" * 60

print(SEPARATOR)
print("  🎙️  SPEECH-TO-REASONING PIPELINE — FULL RESULTS")
print(SEPARATOR)

print("\n  ① AUDIO INPUT")
print(SEP_THIN)
print(f"  File   : {AUDIO_PATH}")

print("\n  ② WHISPER TRANSCRIPTION (ASR Output)")
print(SEP_THIN)
print(f"  Model  : whisper-{WHISPER_MODEL_SIZE}")
print(f"  Text   : {TRANSCRIBED_TEXT}")

print("\n  ③ LLM REASONING OUTPUT")
print(SEP_THIN)
print(f"  Model  : {MODEL_NAME}")
print(f"  Quant  : Dynamic 4-bit (Unsloth)")
print()
print(response)

print()
print(SEPARATOR)
print("  📊 PERFORMANCE METRICS")
print(SEPARATOR)
print(f"  Input tokens    : {num_input_tokens}")
print(f"  Output tokens   : {num_output_tokens}")
print(f"  Inference time  : {inference_time:.2f}s")
print(f"  Throughput      : {tokens_per_sec:.1f} tok/s")
if DEVICE == "cuda":
    peak_vram = torch.cuda.max_memory_allocated() / 1e9
    print(f"  Peak VRAM       : {peak_vram:.2f} GB")
print(SEPARATOR)

# Render nicely in notebook markdown
display(Markdown(f"""
---
### ✅ Pipeline Completed Successfully

| Stage | Detail |
|---|---|
| **ASR Model** | `whisper-{WHISPER_MODEL_SIZE}` |
| **LLM Model** | `{MODEL_NAME}` |
| **Quantization** | Dynamic 4-bit (Unsloth) |
| **Input Tokens** | {num_input_tokens} |
| **Output Tokens** | {num_output_tokens} |
| **Throughput** | {tokens_per_sec:.1f} tok/s |
"""))

---
## 🔄 Step 10: Reusable Pipeline Function
Wrap everything into a clean function you can call with any audio file.

In [ ]:
def speech_to_reasoning(
    audio_path:     str,
    whisper_size:   str  = "small",
    system_prompt:  str  = None,
    max_new_tokens: int  = 512,
    temperature:    float = 0.3,
) -> dict:
    """
    End-to-end Speech-to-Reasoning pipeline.

    Parameters
    ----------
    audio_path      : Path to audio file (WAV / MP3 / M4A / FLAC)
    whisper_size    : Whisper model variant (tiny/base/small/medium/large-v3)
    system_prompt   : Optional custom system instruction for the LLM
    max_new_tokens  : Max tokens to generate
    temperature     : Sampling temperature (lower = more deterministic)

    Returns
    -------
    dict with keys: transcription, response, metrics
    """
    results = {}

    # ── 1. Transcribe ──────────────────────────────────────────────────────
    print(f"[1/3] 🎙️  Loading Whisper '{whisper_size}'...")
    asr = whisper.load_model(whisper_size, device=DEVICE)
    t_asr = time.time()
    result = asr.transcribe(audio_path, fp16=(DEVICE=="cuda"), language="en")
    asr_time = time.time() - t_asr
    transcription = result["text"].strip()
    del asr; gc.collect(); torch.cuda.empty_cache()
    results["transcription"] = transcription
    print(f"     ✅ Transcribed in {asr_time:.1f}s → '{transcription[:80]}...'")

    # ── 2. Build Prompt ────────────────────────────────────────────────────
    sys_msg = system_prompt or SYSTEM_PROMPT
    msgs = [
        {"role": "system", "content": sys_msg},
        {"role": "user",   "content": transcription},
    ]
    prompt = tokenizer.apply_chat_template(
        msgs, tokenize=False, add_generation_prompt=True
    )
    ids = tokenizer(prompt, return_tensors="pt").input_ids.to(DEVICE)

    # ── 3. Generate ────────────────────────────────────────────────────────
    print(f"[2/3] 🤔  Generating reasoning ({max_new_tokens} max tokens)...")
    t_gen = time.time()
    with torch.no_grad():
        out = llm_model.generate(
            ids,
            max_new_tokens    = max_new_tokens,
            temperature       = temperature,
            top_p             = 0.9,
            repetition_penalty= 1.1,
            do_sample         = True,
            use_cache         = True,
            pad_token_id      = tokenizer.eos_token_id,
        )
    gen_time = time.time() - t_gen
    answer = tokenizer.decode(out[0, ids.shape[-1]:], skip_special_tokens=True).strip()
    results["response"] = answer

    # ── 4. Metrics ─────────────────────────────────────────────────────────
    n_out = out.shape[-1] - ids.shape[-1]
    results["metrics"] = {
        "asr_time_s":      round(asr_time, 2),
        "gen_time_s":      round(gen_time, 2),
        "input_tokens":    ids.shape[-1],
        "output_tokens":   n_out,
        "tokens_per_sec":  round(n_out / gen_time, 1),
    }
    print(f"[3/3] ✅  Done in {gen_time:.1f}s ({n_out/gen_time:.1f} tok/s)")
    return results


# ── Demo call ──────────────────────────────────────────────────────────────
print("🚀 Running pipeline with helper function...")
output = speech_to_reasoning(AUDIO_PATH, whisper_size="small")

print("\n" + "═"*60)
print("  TRANSCRIPTION:")
print("─"*60)
print(output["transcription"])
print("\n  REASONING:")
print("─"*60)
print(output["response"])
print("\n  METRICS:")
print("─"*60)
for k, v in output["metrics"].items():
    print(f"  {k:<20}: {v}")
print("═"*60)

---
## 🧪 Step 11: Batch Processing Multiple Audio Files

In [ ]:
from gtts import gTTS

# ── Generate multiple audio queries for batch demo ─────────────────────────
BATCH_QUERIES = [
    ("math_query.mp3",
     "What is the square root of 144, and how do you verify it?"),
    ("logic_query.mp3",
     "All cats are animals. Some animals are pets. Can we conclude that some cats are pets?"),
    ("science_query.mp3",
     "Why does ice float on water? Explain the molecular reasoning."),
]

# Synthesize all audio files
for fname, query in BATCH_QUERIES:
    gTTS(text=query, lang="en").save(fname)
    print(f"✅ Saved: {fname}")

# ── Batch inference ─────────────────────────────────────────────────────────
batch_results = []

print("\n⚡ Running batch inference...\n")
for fname, _ in BATCH_QUERIES:
    print(f"{'─'*55}")
    print(f"🔊 Processing: {fname}")
    r = speech_to_reasoning(fname, whisper_size="small", max_new_tokens=256)
    batch_results.append({"file": fname, **r})
    print(f"\n💬 Response: {r['response'][:200]}...")
    print()

# ── Summary table ──────────────────────────────────────────────────────────
print("\n📊 Batch Summary:")
print(f"{'File':<25} {'In Tok':>7} {'Out Tok':>8} {'tok/s':>7}")
print("─" * 50)
for r in batch_results:
    m = r["metrics"]
    print(f"{r['file']:<25} {m['input_tokens']:>7} {m['output_tokens']:>8} {m['tokens_per_sec']:>7}")
print("─" * 50)

---
## 💾 Step 12: Upload Your Own Audio (Optional)
Run this cell to upload a custom `.wav` or `.mp3` file and run it through the pipeline.

In [ ]:
from google.colab import files

print("📁 Upload your own audio file (WAV, MP3, M4A, FLAC):")
uploaded = files.upload()

for filename in uploaded.keys():
    print(f"\n🔊 Processing uploaded file: {filename}")
    display(Audio(filename))
    result = speech_to_reasoning(filename, whisper_size="small")
    print("\n📝 Transcription:", result["transcription"])
    print("\n🤔 Reasoning:\n",    result["response"])
    print("\n📊 Metrics:",        result["metrics"])

---
## 🧹 Step 13: Cleanup GPU Memory

In [ ]:
print("🧹 Cleaning up GPU memory...")
del llm_model, tokenizer
gc.collect()
torch.cuda.empty_cache()

if DEVICE == "cuda":
    free = torch.cuda.mem_get_info()[0] / 1e9
    print(f"✅ Cleanup complete. Free VRAM: {free:.2f} GB")
else:
    print("✅ Cleanup complete.")

---
## 📚 Notes & Customization Guide

### Switching the LLM
| Model | VRAM Required | Use Case |
|---|---|---|
| `unsloth/Llama-3.2-3B-Instruct-bnb-4bit` | ~3 GB | Fast, lightweight |
| `unsloth/Qwen2.5-7B-Instruct-bnb-4bit` | ~6 GB | Balanced (**default**) |
| `unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit` | ~7 GB | Strong reasoning |
| `unsloth/Qwen2.5-14B-Instruct-bnb-4bit` | ~12 GB | Best accuracy |

### Switching Whisper Size
| Size | VRAM | Speed | WER |
|---|---|---|---|
| `tiny` | ~1 GB | Fastest | ~11% |
| `base` | ~1 GB | Fast | ~8% |
| `small` | ~2 GB | Moderate | ~5% |
| `medium` | ~5 GB | Slow | ~3% |
| `large-v3` | ~10 GB | Slowest | ~2% |

### Tips
- Always **free Whisper from VRAM** before loading the LLM (done automatically in the functions above)
- Use `temperature=0.0` + `do_sample=False` for fully deterministic/greedy output
- For long audio (>30 min), pass `word_timestamps=True` to Whisper for fine-grained alignment
- Unsloth's dynamic 4-bit quantization maintains near-float16 accuracy with ~50% less VRAM